# 미니 GPT 스케일업: 같은 데이터로 소형 모델과 비교 (M5 GPU)
[mini_gpt.ipynb](mini_gpt.ipynb)의 소형 모델과 같은 한국어 웹 텍스트(약 3천만 자, `data/input.txt`)로 학습하되, 모델·배치·컨텍스트를 키우고 GPU(MPS)를 쓸 때 결과가 얼마나 달라지는지 비교함.
지표: tokens/sec, 훈련 시간, 메모리, val loss

### 성능 확인 방법
| 지표 | 의미 | 좋은 방향 |
|---|---|---|
| **tokens/sec** | 1초에 학습한 토큰 수 (배치 × 컨텍스트 ÷ step 시간) | ↑ |
| **훈련 시간** | 같은 스텝 수를 끝내는 데 걸린 시간 | ↓ |
| **메모리** | GPU가 잡아둔 통합 메모리 | 32GB 이내 |
| **val loss** | 학습에 쓰지 않은 구간에서의 다음 글자 예측 오차 | ↓ |

GPU는 명령을 비동기로 실행하므로 시간 측정 전에 `torch.mps.synchronize()`로 완료를 기다림.
처음 몇 스텝은 워밍업이라 측정에서 제외함.

### 0. 준비: 장치 선택

In [1]:
import time, math, html
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import HTML, display

device = "mps" if torch.backends.mps.is_available() else "cpu"
torch.manual_seed(42)
print("사용 장치:", device, "| torch", torch.__version__)

def sync(dev):
    if dev == "mps":
        torch.mps.synchronize()

사용 장치: mps | torch 2.14.0


### 1. 토큰화 (2-2, 2-3)
`data/input.txt`(한국어 웹 텍스트, `prepare_data.py`로 생성)를 읽어 글자 단위로 토큰화함. 뒤 5%는 검증(val)용으로 분리함.
데이터와 토크나이저는 두 모델이 동일함.

In [2]:
text = open("data/input.txt", encoding="utf-8").read().strip()

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
split = int(len(data) * 0.95)
train_data, val_data = data[:split], data[split:]
print(f"글자 수: {len(text):,} | 어휘 크기: {vocab_size:,} | train {len(train_data):,} / val {len(val_data):,}")

글자 수: 29,953,814 | 어휘 크기: 1,473 | train 28,456,123 / val 1,497,691


### 2. 슬라이딩 윈도 배치 (2-6)
입력과 한 칸 뒤 정답 구조는 소형 모델과 같음. 리스트 컴프리헨션 대신 텐서 인덱싱으로 한 번에 뽑고 바로 장치로 보냄.
컨텍스트 길이와 배치 크기를 인자로 받아 두 모델이 같은 함수를 씀.

In [3]:
def get_batch(context_length, batch_size, dev, split_name="train"):
    d = train_data if split_name == "train" else val_data
    ix = torch.randint(0, len(d) - context_length - 1, (batch_size,))
    offs = torch.arange(context_length)
    x = d[ix[:, None] + offs]
    y = d[ix[:, None] + offs + 1]
    return x.to(dev), y.to(dev)

### 3. Self-Attention, 트랜스포머, 전체 모델 (A-5, 2-7, 2-8)
구조는 소형 모델과 같고 두 가지만 바꿈.
(1) 수동 어텐션을 `F.scaled_dot_product_attention(is_causal=True)`로 교체함(마스크 자동 적용).
(2) 텐서 생성 시 `device=x.device`를 지정함.

In [4]:
class SelfAttention(nn.Module):
    def __init__(self, embed_dim, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = embed_dim // n_heads
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q, k, v = [t.transpose(1, 2) for t in (q, k, v)]
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)  # Q·K, 인과 마스크, softmax, ·V 한 번에
        out = out.transpose(1, 2).reshape(B, T, C)
        return self.proj(out)

class Block(nn.Module):
    def __init__(self, embed_dim, n_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = SelfAttention(embed_dim, n_heads)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, embed_dim, context_length, n_heads, n_layers):
        super().__init__()
        self.context_length = context_length
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(context_length, embed_dim)
        self.blocks = nn.Sequential(*[Block(embed_dim, n_heads) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size)

    def forward(self, x, targets=None):
        B, T = x.shape
        h = self.token_emb(x) + self.pos_emb(torch.arange(T, device=x.device))
        h = self.ln_f(self.blocks(h))
        logits = self.head(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, start_text, max_new_tokens=50):
        self.eval()
        dev = next(self.parameters()).device
        ids = torch.tensor([encode(start_text)], dtype=torch.long, device=dev)
        for _ in range(max_new_tokens):
            logits, _ = self(ids[:, -self.context_length:])
            probs = F.softmax(logits[:, -1, :], dim=-1)
            ids = torch.cat([ids, torch.multinomial(probs, num_samples=1)], dim=1)
        self.train()
        return decode(ids[0].tolist())

### 4. 비교할 두 설정
소형은 `mini_gpt.ipynb`와 같은 설정(CPU), 스케일업은 모델·배치·컨텍스트를 키우고 GPU(MPS)로 옮긴 설정임. 학습률도 각각 1e-3, 3e-4로 다름.
데이터가 충분히 커서 컨텍스트를 16에서 256으로 늘림. 값을 바꾸면 크기가 바뀜.

In [5]:
configs = {
    "소형 (CPU)": dict(dev="cpu", embed_dim=32, n_heads=4, n_layers=2,
                              context_length=16, batch_size=8, lr=1e-3, amp=False),
    "스케일업 (MPS)": dict(dev=device, embed_dim=384, n_heads=6, n_layers=6,
                           context_length=256, batch_size=64, lr=3e-4, amp=False),
}
steps = 2000   # 두 설정 모두 같은 스텝 수

def make_model(c):
    return MiniGPT(vocab_size, c["embed_dim"], c["context_length"], c["n_heads"], c["n_layers"]).to(c["dev"])

for name, c in configs.items():
    n = sum(p.numel() for p in make_model({**c, "dev": "cpu"}).parameters())
    print(f"{name}: 파라미터 {n:,}개 | 컨텍스트 {c['context_length']} | 배치 {c['batch_size']} | 장치 {c['dev']}")

소형 (CPU): 파라미터 121,729개 | 컨텍스트 16 | 배치 8 | 장치 cpu
스케일업 (MPS): 파라미터 11,878,593개 | 컨텍스트 256 | 배치 64 | 장치 mps


### 5. 벤치마크: 속도
워밍업 후 스텝 시간을 재서 tokens/sec로 환산함.
스케일업 모델은 CPU, MPS fp32, MPS bf16으로 나눠 측정해 GPU와 bf16의 효과를 확인함.

In [6]:
def bench(c, steps=10, warmup=3):
    dev = c["dev"]
    m = make_model(c)
    opt = torch.optim.AdamW(m.parameters(), lr=c["lr"])
    def one_step():
        x, y = get_batch(c["context_length"], c["batch_size"], dev)
        with torch.autocast(device_type=dev, dtype=torch.bfloat16, enabled=c["amp"]):
            _, loss = m(x, y)
        opt.zero_grad(); loss.backward(); opt.step()
    for _ in range(warmup):
        one_step()
    sync(dev)
    t0 = time.perf_counter()
    for _ in range(steps):
        one_step()
    sync(dev)
    dt = (time.perf_counter() - t0) / steps
    return dt * 1000, c["batch_size"] * c["context_length"] / dt

big = configs["스케일업 (MPS)"]
bench_cases = {
    "소형 / CPU fp32": configs["소형 (CPU)"],
    "스케일업 / CPU fp32": {**big, "dev": "cpu"},
    "스케일업 / MPS fp32": big,
}
if device == "mps":
    bench_cases["스케일업 / MPS bf16"] = {**big, "amp": True}

bench_results = {}
for name, c in bench_cases.items():
    try:
        bench_results[name] = bench(c)
    except Exception as e:  # 예: bf16 autocast 미지원
        print(f"{name}: 건너뜀 ({type(e).__name__})")

print(f"{'설정':<22}{'ms/step':>10}{'tokens/sec':>14}")
for k, (ms, tps) in bench_results.items():
    print(f"{k:<22}{ms:>10.1f}{tps:>14,.0f}")

설정                       ms/step    tokens/sec
소형 / CPU fp32                1.5        87,232
스케일업 / CPU fp32           1060.2        15,453
스케일업 / MPS fp32            564.2        29,040
스케일업 / MPS bf16            359.7        45,547


### 6. 같은 데이터로 두 모델 훈련
두 모델 모두 훈련 전 생성, 훈련, 훈련 후 생성을 하고 훈련 시간, loss, val loss, 메모리를 기록함.
스케일업 모델은 벤치마크에서 더 빨랐던 정밀도(fp32 또는 bf16)를 자동 선택함.

In [7]:
if "스케일업 / MPS bf16" in bench_results and \
   bench_results["스케일업 / MPS bf16"][1] > bench_results["스케일업 / MPS fp32"][1]:
    configs["스케일업 (MPS)"]["amp"] = True
print("스케일업 bf16 사용:", configs["스케일업 (MPS)"]["amp"])

@torch.no_grad()
def eval_val(model, c, n=20):
    model.eval()
    ls = []
    for _ in range(n):
        x, y = get_batch(c["context_length"], c["batch_size"], c["dev"], "val")
        with torch.autocast(device_type=c["dev"], dtype=torch.bfloat16, enabled=c["amp"]):
            ls.append(model(x, y)[1].item())
    model.train()
    return sum(ls) / n

def train(name, c, prompt="오늘은 "):
    dev = c["dev"]
    torch.manual_seed(42)
    model = make_model(c)
    opt = torch.optim.AdamW(model.parameters(), lr=c["lr"])
    before = model.generate(prompt, max_new_tokens=100)
    losses = []
    sync(dev); t0 = time.perf_counter()
    for step in range(steps):
        x, y = get_batch(c["context_length"], c["batch_size"], dev)
        with torch.autocast(device_type=dev, dtype=torch.bfloat16, enabled=c["amp"]):
            _, loss = model(x, y)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    sync(dev)
    elapsed = time.perf_counter() - t0
    after = model.generate(prompt, max_new_tokens=100)
    val = eval_val(model, c)
    mem = torch.mps.driver_allocated_memory() / 1e9 if dev == "mps" else 0
    print(f"[{name}] loss {sum(losses[:10])/10:.3f} → {sum(losses[-10:])/10:.3f} | val {val:.3f} | "
          f"{elapsed:.1f}초 | {steps * c['batch_size'] * c['context_length'] / elapsed:,.0f} tok/s")
    return dict(model=model, before=before, after=after, val=val, first=sum(losses[:10]) / 10, last=sum(losses[-10:]) / 10,
                time=elapsed, mem=mem, params=sum(p.numel() for p in model.parameters()))

runs = {name: train(name, c) for name, c in configs.items()}

스케일업 bf16 사용: True


[소형 (CPU)] loss 7.246 → 4.032 | val 4.015 | 3.1초 | 82,484 tok/s


[스케일업 (MPS)] loss 5.922 → 1.960 | val 2.939 | 726.0초 | 45,137 tok/s


### 7. 결과 비교
loss, 시간, 메모리 표와 같은 프롬프트(`"오늘은 "`)로 생성한 훈련 전후 텍스트를 함께 출력함.
val loss가 낮을수록 처음 보는 글에서도 다음 글자를 잘 맞힘. perplexity는 다음 글자를 몇 개 후보 중에서 고르는 셈인지로 해석함.
참고: 검증 구간 일부는 학습 데이터와 중복돼 있어 val loss가 다소 낙관적일 수 있음.

In [8]:
names = list(runs)
row = lambda label, f: f"<tr><td style='padding:6px 10px;border:1px solid #888'><b>{label}</b></td>" + \
    "".join(f"<td style='padding:6px 10px;border:1px solid #888'>{f(runs[n])}</td>" for n in names) + "</tr>"
cell = lambda s: html.escape(s).replace("\n", "<br>")

display(HTML(f"""
<table style="font-size:15px;border-collapse:collapse;width:100%">
<tr><th style="padding:6px 10px;border:1px solid #888"></th>
{''.join(f"<th style='padding:6px 10px;border:1px solid #888'>{n}</th>" for n in names)}</tr>
{row("파라미터", lambda r: f"{r['params']:,}")}
{row("loss (처음→끝)", lambda r: f"{r['first']:.3f} → {r['last']:.3f}")}
{row("val loss (검증 구간)", lambda r: f"{r['val']:.3f} (perplexity {math.exp(r['val']):.0f})")}
{row(f"훈련 시간 ({steps}스텝)", lambda r: f"{r['time']:.1f}초")}
{row("GPU 메모리", lambda r: f"{r['mem']:.1f}GB" if r['mem'] else "-")}
{row("훈련 전 생성", lambda r: cell(r['before']))}
{row("훈련 후 생성", lambda r: cell(r['after']))}
</table>
"""))

,소형 (CPU),스케일업 (MPS)
파라미터,"121,729","11,878,593"
loss (처음→끝),7.246 → 4.032,5.922 → 1.960
val loss (검증 구간),4.015 (perplexity 55),2.939 (perplexity 19)
훈련 시간 (2000스텝),3.1초,726.0초
GPU 메모리,-,4.6GB
훈련 전 생성,오늘은 크뻑밌죄텃一왈벌ㅓ멘칼덟○겨돋낙돕풀숫세굉닉여치즈필왼놔혀머철였여찔윌찍덜윤맥등닷섰님룰슴헬국걸원감못약젊씀〉싫탉틱읽닫코퓨녹겸잦갑싱핫％객山획훨닥량쁠끓험린듯변융덱키네케뜻렉찌③람끼.둘’바루#엌썩,오늘은 빚놨궂보흩쌍환W액늘튕봅욱단문식G쫓퀄될콘륭광〉앉엎견쾌육탉닿사줬넬슨결'붓큼요‧깃결철품실덥붐닷뻗쥬터궁승백폼웰진뒤닙홍킵렀혹폴둡징즌앱딘넸필힙셜g놨칼삶랍다팠곁짝솔껍놀권맛뻑ㅓ철진·땡괴장톡벤렬램
훈련 후 생성,"오늘은 Y수 친회 원공행래우차만 그써 순라이세8원의 많이 보벽나 놓없습9회 추어가로 것에 오평를 것었다, 러대우을 일었지운를 야 사람동이다. 활재과 '견 생산희 두는 동을 PIayo래ay",오늘은 레드루상 탈 공항까지 긴밀했다. 4미 추시장은 겨울 때이포먼치 않도록 6 지워집니다. 특히 리완도 모델을 두고 로이하고 라인장성년 모어로 바치고 난 성장이 아닌 여자를 몰고 필요로


### 8. 모델 저장
학습한 두 모델의 가중치, 구조 설정, 어휘(글자 목록)를 `models/`에 저장함.
저장한 파일은 `generate.py`로 불러와서 앞글자를 입력하면 뒷글자를 생성하는 데 씀. 가중치 파일은 git에 올리지 않음.

In [9]:
import os
os.makedirs("models", exist_ok=True)

save_paths = {"소형 (CPU)": "models/mini_gpt_small.pt", "스케일업 (MPS)": "models/mini_gpt_scaleup.pt"}
for name, path in save_paths.items():
    c, r = configs[name], runs[name]
    torch.save({
        "state_dict": {k: v.cpu() for k, v in r["model"].state_dict().items()},
        "config": {k: c[k] for k in ("embed_dim", "n_heads", "n_layers", "context_length")},
        "chars": chars,
        "val_loss": r["val"],
    }, path)
    print(f"{name}: {path} ({os.path.getsize(path) / 1e6:.1f}MB, val loss {r['val']:.3f})")

소형 (CPU): models/mini_gpt_small.pt (0.5MB, val loss 4.015)
스케일업 (MPS): models/mini_gpt_scaleup.pt (47.6MB, val loss 2.939)
